# Module 2.4 - Vector Databases (ChromaDB)

**Reference:** [`04-vector-databases.md`](./04-vector-databases.md)

## What you'll do in this notebook

- Create a persistent ChromaDB collection on local disk.
- Add documents with OpenAI embeddings.
- Perform similarity search and interpret the scores.
- Use metadata filters to scope searches.

## Setup

ChromaDB is local and persistent by default. The `PersistentClient` writes files under `path=...` so your collection survives a kernel restart.

In [ ]:
import os
import shutil
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"
CHROMA_PATH = "./chroma_store_m2"

# Start fresh each time this notebook runs.
if os.path.exists(CHROMA_PATH):
    shutil.rmtree(CHROMA_PATH)

chroma = chromadb.PersistentClient(path=CHROMA_PATH)
print(f"OK: ChromaDB persistent client at {CHROMA_PATH}")

## Exercise 1 - Create a collection and add documents

ChromaDB organises data into collections. Each item has an `id`, an `embedding`, a `document` (the raw text), and optional `metadata`.

In [ ]:
DOCS = [
    "Python was created by Guido van Rossum and first released in 1991.",
    "JavaScript is the language of the web, running in every browser.",
    "Rust is a systems language focused on memory safety without a garbage collector.",
    "Python lists are ordered, mutable sequences.",
    "A Python dictionary stores key-value pairs with O(1) average lookup.",
    "Exception handling in Python uses try/except/finally.",
]

METAS = [
    {"source": "python_history.md", "topic": "history"},
    {"source": "javascript_intro.md", "topic": "history"},
    {"source": "rust_intro.md", "topic": "history"},
    {"source": "python_collections.md", "topic": "data_structures"},
    {"source": "python_collections.md", "topic": "data_structures"},
    {"source": "python_errors.md", "topic": "error_handling"},
]

def embed_batch(texts: list[str]) -> list[list[float]]:
    # TODO: call client.embeddings.create once with input=texts and return
    # a list of embedding vectors (one per text, in order).
    raise NotImplementedError

collection = chroma.create_collection(name="programming")

# TODO:
# 1. Build embeddings for DOCS using embed_batch.
# 2. Call collection.add(ids=..., embeddings=..., documents=..., metadatas=...).
#    Use ids like "doc_0", "doc_1", ...

print(f"Collection now has {collection.count()} documents")

## Exercise 2 - Similarity search

Embed a user query and ask the collection for the nearest matches.

In [ ]:
def search(query: str, top_k: int = 3) -> dict:
    # TODO:
    # 1. Embed query with embed_batch([query])[0].
    # 2. Call collection.query(query_embeddings=[...], n_results=top_k).
    # 3. Return the results dict unchanged.
    raise NotImplementedError

results = search("Who invented Python?")
for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"[{dist:.3f}] ({meta['source']}) {doc}")

ChromaDB's default similarity metric is cosine *distance*, so **smaller** values mean more similar. Sanity-check that the Python-history document comes out on top for a "who invented Python?" query.

## Exercise 3 - Metadata filtering

Filter the search so it only considers documents whose `topic` field matches a target.

In [ ]:
def search_with_filter(query: str, where: dict, top_k: int = 3) -> dict:
    # TODO: same as search(), but also pass where=where to collection.query.
    # Example: where={"topic": "data_structures"}.
    raise NotImplementedError

results = search_with_filter("How do I store key-value pairs?", where={"topic": "data_structures"})
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"({meta['source']}) {doc}")

**Try it:** ask "Who invented Python?" with `where={"topic": "data_structures"}`. What happens? Why?

## Bonus - Persistence

Stop and restart this notebook's kernel. Run only the *Setup* cell (but remove the `shutil.rmtree` line so it doesn't wipe the store) and then `chroma.get_or_create_collection("programming").count()`. You should see the documents still there - that's what the `Persistent` in `PersistentClient` buys you.

## What's next

In `05-building-a-rag-chatbot.ipynb` we combine chunking, embedding, ChromaDB search, and LLM generation into one `RAGChatbot` class — the first end-to-end chatbot that answers questions from your own documents.